In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import os
import time
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import Xception
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

from sklearn.metrics import classification_report, confusion_matrix, recall_score, fbeta_score

In [3]:
TRAIN_PATH = "/content/drive/MyDrive/Cataract/Data/Train"
TEST_PATH  = "/content/drive/MyDrive/Cataract/Data/Test"

MODEL_PATH = "/content/drive/MyDrive/Cataract/Xception_FineTune.h5"

In [4]:
print("Train folders:", os.listdir(TRAIN_PATH))
print("Test folders:", os.listdir(TEST_PATH))

Train folders: ['Cataract', 'Normal', 'Not Eye']
Test folders: ['Cataract', 'Normal', 'Not Eye']


In [5]:
datagen = ImageDataGenerator(
    rescale = 1. / 255,
    validation_split = 0.2,
    horizontal_flip = True,
    vertical_flip = True
)

In [6]:
train_it = datagen.flow_from_directory(
    TRAIN_PATH,
    target_size = (224, 224),
    color_mode = 'rgb',
    class_mode = 'categorical',
    batch_size = 32,
    subset = "training"
)

val_it = datagen.flow_from_directory(
    TRAIN_PATH,
    target_size = (224, 224),
    color_mode = 'rgb',
    class_mode = 'categorical',
    batch_size = 32,
    subset = "validation"
)

test_it = datagen.flow_from_directory(
    TEST_PATH,
    target_size = (224, 224),
    color_mode = 'rgb',
    class_mode = 'categorical',
    batch_size = 32,
    shuffle = False
)

print("Class indices:", train_it.class_indices)

Found 8896 images belonging to 3 classes.
Found 2221 images belonging to 3 classes.
Found 2552 images belonging to 3 classes.
Class indices: {'Cataract': 0, 'Normal': 1, 'Not Eye': 2}


In [7]:
base_model = Xception(
    weights = 'imagenet',
    input_shape = (224, 224, 3),
    include_top = False
)

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [8]:
ocl1 = Conv2D(32, (3, 3), activation='relu')(base_model.output)
bn1 = BatchNormalization()(ocl1)
mp1 = MaxPooling2D(pool_size=(2, 2))(bn1)
do1 = Dropout(0.17)(mp1)

ocl2 = Conv2D(64, (2, 2), activation='relu')(do1)
bn2 = BatchNormalization()(ocl2)

al1 = GlobalAveragePooling2D()(bn2)

fc1 = Dense(64, activation='relu')(al1)
fc2 = Dense(32, activation='relu')(fc1)
fc3 = Dense(32, activation='relu')(fc2)

al2 = BatchNormalization()(fc3)
all2 = Dropout(0.3)(al2)

output = Dense(3, activation='softmax', name='preds')(all2)

In [9]:
Cataract_Model = Model(
    inputs = base_model.input,
    outputs = output
)

Cataract_Model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 111, 111,  │        864 │ input_layer[0][0] │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_bn     │ (None, 111, 111,  │        128 │ block1_conv1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_act    │ (None, 111, 111,  │          0 │ block1_conv1_bn[… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 109, 109,  │     18,432 │ block1_conv1_act… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_bn     │ (None, 109, 109,  │        256 │ block1_conv2[0][… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_act    │ (None, 109, 109,  │          0 │ block1_conv2_bn[… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1     │ (None, 109, 109,  │      8,768 │ block1_conv2_act… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1_bn  │ (None, 109, 109,  │        512 │ block2_sepconv1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_act │ (None, 109, 109,  │          0 │ block2_sepconv1_… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2     │ (None, 109, 109,  │     17,536 │ block2_sepconv2_… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_bn  │ (None, 109, 109,  │        512 │ block2_sepconv2[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 55, 55,    │      8,192 │ block1_conv2_act… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 55, 55,    │          0 │ block2_sepconv2_… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 55, 55,    │        512 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 55, 55,    │          0 │ block2_pool[0][0… │
│                     │ 128)              │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_sepconv1_act │ (None, 55, 55,    │          0 │ add[0][0]       

 Total params: 21,467,499 (81.89 MB)

 Trainable params: 21,412,715 (81.68 MB)

 Non-trainable params: 54,784 (214.00 KB)

In [10]:
Cataract_Model.compile(
    optimizer = Adam(learning_rate=0.0001),
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

In [ ]:
for ix in range(132):
    Cataract_Model.layers[ix].trainable = False

In [ ]:
for i, layer in enumerate(Cataract_Model.layers):
    print(i, layer.name, layer.trainable)

0 input_layer False
1 block1_conv1 False
2 block1_conv1_bn False
3 block1_conv1_act False
4 block1_conv2 False
5 block1_conv2_bn False
6 block1_conv2_act False
7 block2_sepconv1 False
8 block2_sepconv1_bn False
9 block2_sepconv2_act False
10 block2_sepconv2 False
11 block2_sepconv2_bn False
12 conv2d False
13 block2_pool False
14 batch_normalization False
15 add False
16 block3_sepconv1_act False
17 block3_sepconv1 False
18 block3_sepconv1_bn False
19 block3_sepconv2_act False
20 block3_sepconv2 False
21 block3_sepconv2_bn False
22 conv2d_1 False
23 block3_pool False
24 batch_normalization_1 False
25 add_1 False
26 block4_sepconv1_act False
27 block4_sepconv1 False
28 block4_sepconv1_bn False
29 block4_sepconv2_act False
30 block4_sepconv2 False
31 block4_sepconv2_bn False
32 conv2d_2 False
33 block4_pool False
34 batch_normalization_2 False
35 add_2 False
36 block5_sepconv1_act False
37 block5_sepconv1 False
38 block5_sepconv1_bn False
39 block5_sepconv2_act False
40 block5_sepconv2 F

In [ ]:
mc = ModelCheckpoint(
    MODEL_PATH,
    monitor = 'val_accuracy',
    mode = 'max',
    verbose = 1,
    save_best_only = True
)

In [ ]:
history = Cataract_Model.fit(
    train_it,
    epochs = 20,
    validation_data = val_it,
    callbacks = [mc]
)

Epoch 1/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.6731 - loss: 0.7995
Epoch 1: val_accuracy improved from None to 0.96578, saving model to /content/drive/MyDrive/Cataract/Xception_FineTune.h5



Epoch 1: finished saving model to /content/drive/MyDrive/Cataract/Xception_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 3302s 12s/step - accuracy: 0.8076 - loss: 0.5135 - val_accuracy: 0.9658 - val_loss: 0.1610
Epoch 2/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step - accuracy: 0.9357 - loss: 0.2201
Epoch 2: val_accuracy improved from 0.96578 to 0.97073, saving model to /content/drive/MyDrive/Cataract/Xception_FineTune.h5



Epoch 2: finished saving model to /content/drive/MyDrive/Cataract/Xception_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 72s 258ms/step - accuracy: 0.9420 - loss: 0.1968 - val_accuracy: 0.9707 - val_loss: 0.0882
Epoch 3/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step - accuracy: 0.9550 - loss: 0.1501
Epoch 3: val_accuracy improved from 0.97073 to 0.97299, saving model to /content/drive/MyDrive/Cataract/Xception_FineTune.h5



Epoch 3: finished saving model to /content/drive/MyDrive/Cataract/Xception_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 80s 249ms/step - accuracy: 0.9548 - loss: 0.1466 - val_accuracy: 0.9730 - val_loss: 0.0805
Epoch 4/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step - accuracy: 0.9713 - loss: 0.1037
Epoch 4: val_accuracy improved from 0.97299 to 0.98649, saving model to /content/drive/MyDrive/Cataract/Xception_FineTune.h5



Epoch 4: finished saving model to /content/drive/MyDrive/Cataract/Xception_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 72s 256ms/step - accuracy: 0.9707 - loss: 0.1013 - val_accuracy: 0.9865 - val_loss: 0.0515
Epoch 5/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - accuracy: 0.9763 - loss: 0.0838
Epoch 5: val_accuracy did not improve from 0.98649
278/278 ━━━━━━━━━━━━━━━━━━━━ 75s 269ms/step - accuracy: 0.9755 - loss: 0.0824 - val_accuracy: 0.9860 - val_loss: 0.0453
Epoch 6/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step - accuracy: 0.9811 - loss: 0.0705
Epoch 6: val_accuracy did not improve from 0.98649
278/278 ━━━━━━━━━━━━━━━━━━━━ 67s 242ms/step - accuracy: 0.9792 - loss: 0.0715 - val_accuracy: 0.9842 - val_loss: 0.0467
Epoch 7/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - accuracy: 0.9837 - loss: 0.0573
Epoch 7: val_accuracy did not improve from 0.98649
278/278 ━━━━━━━━━━━━━━━━━━━━ 68s 246ms/step - accuracy: 0.9824 - loss: 0.0592 - val_accuracy: 0.9856 - val_loss: 0.0411
Epoch 8/20
278/


Epoch 9: finished saving model to /content/drive/MyDrive/Cataract/Xception_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 70s 251ms/step - accuracy: 0.9875 - loss: 0.0476 - val_accuracy: 0.9869 - val_loss: 0.0430
Epoch 10/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - accuracy: 0.9876 - loss: 0.0451
Epoch 10: val_accuracy improved from 0.98694 to 0.98964, saving model to /content/drive/MyDrive/Cataract/Xception_FineTune.h5



Epoch 10: finished saving model to /content/drive/MyDrive/Cataract/Xception_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 71s 253ms/step - accuracy: 0.9870 - loss: 0.0456 - val_accuracy: 0.9896 - val_loss: 0.0295
Epoch 11/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step - accuracy: 0.9885 - loss: 0.0451
Epoch 11: val_accuracy did not improve from 0.98964
278/278 ━━━━━━━━━━━━━━━━━━━━ 70s 249ms/step - accuracy: 0.9861 - loss: 0.0496 - val_accuracy: 0.9887 - val_loss: 0.0360
Epoch 12/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step - accuracy: 0.9861 - loss: 0.0469
Epoch 12: val_accuracy improved from 0.98964 to 0.99190, saving model to /content/drive/MyDrive/Cataract/Xception_FineTune.h5



Epoch 12: finished saving model to /content/drive/MyDrive/Cataract/Xception_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 69s 249ms/step - accuracy: 0.9877 - loss: 0.0446 - val_accuracy: 0.9919 - val_loss: 0.0254
Epoch 13/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step - accuracy: 0.9909 - loss: 0.0357
Epoch 13: val_accuracy did not improve from 0.99190
278/278 ━━━━━━━━━━━━━━━━━━━━ 79s 238ms/step - accuracy: 0.9899 - loss: 0.0362 - val_accuracy: 0.9901 - val_loss: 0.0335
Epoch 14/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step - accuracy: 0.9903 - loss: 0.0338
Epoch 14: val_accuracy did not improve from 0.99190
278/278 ━━━━━━━━━━━━━━━━━━━━ 67s 240ms/step - accuracy: 0.9913 - loss: 0.0311 - val_accuracy: 0.9914 - val_loss: 0.0243
Epoch 15/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step - accuracy: 0.9880 - loss: 0.0362
Epoch 15: val_accuracy did not improve from 0.99190
278/278 ━━━━━━━━━━━━━━━━━━━━ 66s 238ms/step - accuracy: 0.9900 - loss: 0.0323 - val_accuracy: 0.9784 - val_loss: 0.0631
Epoch 16


Epoch 16: finished saving model to /content/drive/MyDrive/Cataract/Xception_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 83s 243ms/step - accuracy: 0.9897 - loss: 0.0338 - val_accuracy: 0.9932 - val_loss: 0.0217
Epoch 17/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step - accuracy: 0.9921 - loss: 0.0289
Epoch 17: val_accuracy did not improve from 0.99325
278/278 ━━━━━━━━━━━━━━━━━━━━ 67s 240ms/step - accuracy: 0.9922 - loss: 0.0275 - val_accuracy: 0.9923 - val_loss: 0.0309
Epoch 18/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step - accuracy: 0.9936 - loss: 0.0245
Epoch 18: val_accuracy did not improve from 0.99325
278/278 ━━━━━━━━━━━━━━━━━━━━ 67s 240ms/step - accuracy: 0.9931 - loss: 0.0262 - val_accuracy: 0.9910 - val_loss: 0.0271
Epoch 19/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step - accuracy: 0.9959 - loss: 0.0133
Epoch 19: val_accuracy did not improve from 0.99325
278/278 ━━━━━━━━━━━━━━━━━━━━ 67s 239ms/step - accuracy: 0.9943 - loss: 0.0182 - val_accuracy: 0.9739 - val_loss: 0.0888
Epoch 20

In [11]:
best_model = tf.keras.models.load_model(MODEL_PATH)
print("Best model loaded:", MODEL_PATH)

Best model loaded: /content/drive/MyDrive/Cataract/Xception_FineTune.h5


In [ ]:
test_loss, test_accuracy = best_model.evaluate(test_it)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

80/80 ━━━━━━━━━━━━━━━━━━━━ 690s 8s/step - accuracy: 0.9933 - loss: 0.0183
Test Loss: 0.018285095691680908
Test Accuracy: 0.9933385848999023


In [ ]:
test_it.reset()

y_pred_probs = best_model.predict(test_it)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_it.classes

recall = recall_score(y_true, y_pred, average='weighted')
f2 = fbeta_score(y_true, y_pred, beta=2, average='weighted')

print("Recall:", recall)
print("F2 Score:", f2)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=list(test_it.class_indices.keys())))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

80/80 ━━━━━━━━━━━━━━━━━━━━ 26s 272ms/step
Recall: 0.9949059561128527
F2 Score: 0.994902027851854

Classification Report:
              precision    recall  f1-score   support

    Cataract       1.00      0.99      0.99       800
      Normal       0.99      1.00      0.99       800
     Not Eye       1.00      1.00      1.00       952

    accuracy                           0.99      2552
   macro avg       0.99      0.99      0.99      2552
weighted avg       0.99      0.99      0.99      2552


Confusion Matrix:
[[790  10   0]
 [  1 799   0]
 [  1   1 950]]


In [12]:
import time
import numpy as np
import os

test_it.reset()
single_image_input = next(iter(test_it))[0][:1]
print(f"Input shape: {single_image_input.shape}  ← batch_size=1 (single image)")

print("Running warm-up inferences...")
for _ in range(10):
    best_model.predict(single_image_input, verbose=0)

N = 100
times = []
for _ in range(N):
    start = time.perf_counter()
    best_model.predict(single_image_input, verbose=0)
    end   = time.perf_counter()
    times.append((end - start) * 1000)

# ── Save 100 values to Drive ──────────────────────────────
np.save("/content/drive/MyDrive/Cataract/latency_Xception.npy", np.array(times))
print("✅ Saved latency_Xception.npy")

latency_mean = np.mean(times)
latency_std  = np.std(times)
latency_p95  = np.percentile(times, 95)
model_size_mb = os.path.getsize("/content/drive/MyDrive/Cataract/Xception_FineTune.h5") / (1024*1024)

print("\n" + "="*55)
print("  Xception — Single-Image Latency Report")
print("="*55)
print(f"  Average Latency : {latency_mean:.2f} ± {latency_std:.2f} ms")
print(f"  P95 Latency     : {latency_p95:.2f} ms")
print(f"  Model File Size : {model_size_mb:.2f} MB")
print(f"  Benchmark Runs  : {N}")
print(f"  Input Shape     : {single_image_input.shape}")
print(f"  Hardware        : Google Colab T4 GPU")
print("="*55)

Input shape: (1, 224, 224, 3)  ← batch_size=1 (single image)
Running warm-up inferences...
✅ Saved latency_Xception.npy

  Xception — Single-Image Latency Report
  Average Latency : 104.28 ± 20.85 ms
  P95 Latency     : 140.29 ms
  Model File Size : 86.99 MB
  Benchmark Runs  : 100
  Input Shape     : (1, 224, 224, 3)
  Hardware        : Google Colab T4 GPU
